# 职业分类 - 三位码（10100-70400）
基于阿里通义千问API，将识别结果中的职业描述匹配到《中国职业大典》三位码分类（78类）。

非从业人员（profession为"未知"）沿用一位码的 0/9/10 分类。

输入：`结果/1.识别结果/` 下的人头计数表和独立职业表
输出：`结果/2.分类结果/三位码/` 下的已分类表格

In [ ]:
import pandas as pd
import json
import os
import time
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen-plus-2025-12-01")

if not API_KEY:
    raise ValueError("请在 2.职业分类/.env 文件中设置 API_KEY")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

PROJECT_ROOT = Path("..").resolve()
INPUT_DIR = PROJECT_ROOT / "结果" / "1.识别结果"
OUTPUT_DIR = PROJECT_ROOT / "结果" / "2.分类结果" / "三位码"
CLASS_FILE_3 = Path("职业分类（三位码）.xlsx")
CLASS_FILE_1 = Path("职业分类.xlsx")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"输入目录: {INPUT_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

In [ ]:
# 加载三位码分类表（78类，代码10100-70400）
df_class3 = pd.read_excel(CLASS_FILE_3)

# 加载一位码分类表（用于非从业人员的0/9/10）
df_class1 = pd.read_excel(CLASS_FILE_1)
if 10 not in df_class1['代码'].values:
    new_row = pd.DataFrame([{
        '代码': 10, '分类名称': '非从业人员-其他',
        '分类内容': '无法归入学生、儿童或家庭角色的其他非从业人员，或身份不明者。'
    }])
    df_class1 = pd.concat([df_class1, new_row], ignore_index=True)

# 构建三位码全量分类参考字符串
class_info_3digit = ""
for _, row in df_class3.iterrows():
    content = str(row.get('分类内容', ''))[:60]
    class_info_3digit += f"代码[{row['代码']}] {row['分类名称']}: {content}\n"

# 构建非从业人员限制分类（沿用一位码0/9/10）
restricted_codes = [0, 9, 10]
class_info_restricted = ""
for _, row in df_class1[df_class1['代码'].isin(restricted_codes)].iterrows():
    content = str(row.get('分类内容', ''))[:50]
    class_info_restricted += f"代码[{row['代码']}] {row['分类名称']}: {content}\n"

# 合并代码→名称映射（三位码 + 一位码的0/9/10）
code_to_name = df_class3.set_index('代码')['分类名称'].to_dict()
for _, row in df_class1[df_class1['代码'].isin(restricted_codes)].iterrows():
    code_to_name[row['代码']] = row['分类名称']

print(f"三位码分类表: {len(df_class3)} 类")
print(f"非从业人员分类: {restricted_codes}")
print(f"\n三位码分类表前5条:")
print(class_info_3digit[:300] + "...")

In [ ]:
def classify_batch_3digit(descriptions: list, class_info_str: str, restricted: bool = False) -> dict:
    """调用大模型对一批描述进行三位码职业分类"""
    if restricted:
        constraint = """【重要限制】这组数据通过视觉无法确认职业，属于非从业人员场景。
请仅从以下三个类别中选择，严禁归入其他职业类：
- 0 (学生/儿童)
- 9 (家庭角色，如母亲、父亲、祖辈)
- 10 (其他非从业人员，或无法判断的非职业身份)"""
        example = '{"背书包的小学生": 0, "做饭的母亲": 9, "路人": 10}'
    else:
        constraint = """请根据描述归入最合适的三位码类别。
注意：三位码代码为5位数字（如10100、20200等），请务必使用完整的5位代码。
如果描述的是学生、儿童，请归入代码0；家庭角色归入代码9。"""
        example = '{"正在讲课的老师": 20800, "警察": 30100, "医生": 20600}'

    prompt = f"""任务：请将以下人物描述归类到《职业分类大典》的对应类别中。

参考分类标准：
{class_info_str}

{constraint}

待分类的描述列表：
{json.dumps(descriptions, ensure_ascii=False)}

要求：
1. 严格输出JSON格式，Key为原描述，Value为对应的"代码"（数字）。
2. 不要输出任何解释性文字。

输出示例：
{example}"""

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"  API调用出错: {e}")
        return {}

print("三位码分类函数定义完成")

In [ ]:
def process_file_3digit(file_path: Path, output_path: Path):
    """处理单个识别结果文件，进行三位码分类"""
    print(f"\n处理: {file_path.name}")
    df = pd.read_excel(file_path)

    if df.empty:
        print("  空文件，跳过")
        return

    mapping_dict = {}
    BATCH_SIZE = 30  # 三位码类别多，batch小一些保证准确率

    # 轨道A：profession != "未知" → 用profession匹配三位码分类表
    known_mask = df['profession'] != '未知'
    descs_known = df.loc[known_mask, 'profession'].dropna().unique().tolist()
    if descs_known:
        print(f"  轨道A (已知职业→三位码): {len(descs_known)} 个描述")
        for i in range(0, len(descs_known), BATCH_SIZE):
            batch = descs_known[i:i+BATCH_SIZE]
            res = classify_batch_3digit(batch, class_info_3digit, restricted=False)
            mapping_dict.update(res)
            time.sleep(0.5)

    # 轨道B：profession == "未知" → 用identifier匹配一位码0/9/10
    unknown_mask = df['profession'] == '未知'
    descs_unknown = df.loc[unknown_mask, 'identifier'].dropna().unique().tolist()
    if descs_unknown:
        print(f"  轨道B (未知职业→一位码0/9/10): {len(descs_unknown)} 个描述")
        for i in range(0, len(descs_unknown), BATCH_SIZE):
            batch = descs_unknown[i:i+BATCH_SIZE]
            res = classify_batch_3digit(batch, class_info_restricted, restricted=True)
            mapping_dict.update(res)
            time.sleep(0.5)

    # 回填
    def apply_mapping(row):
        key = row['identifier'] if row['profession'] == '未知' else row['profession']
        return mapping_dict.get(key, '匹配失败')

    df['职业分类代码'] = df.apply(apply_mapping, axis=1)
    df['职业分类名称'] = df['职业分类代码'].map(code_to_name)

    df.to_excel(output_path, index=False)
    print(f"  已保存: {output_path.name}")
    print(f"  匹配成功: {(df['职业分类代码'] != '匹配失败').sum()}/{len(df)}")

print("处理函数定义完成")

## 执行三位码分类

In [ ]:
# 扫描识别结果目录，处理所有Excel文件
files = list(INPUT_DIR.glob("*.xlsx"))
print(f"发现 {len(files)} 个识别结果文件:\n")
for f in files:
    print(f"  - {f.name}")

print(f"\n{'='*50}")

for file_path in files:
    output_name = f"已分类_三位码_{file_path.name}"
    output_path = OUTPUT_DIR / output_name
    process_file_3digit(file_path, output_path)
    time.sleep(0.5)

print(f"\n{'='*50}")
print("所有文件三位码分类完毕！")